# 🎭 Real-Time AI Avatar Streaming Server
### RunPod Notebook — Port 7865

**Pipeline:**
```
Browser Mic → WebSocket → Moshi → Bridge → AvatarForcing → Browser (A+V)
```

**Architecture:** 7 concurrent async tasks per session:
- `audio_receive`  — browser mic PCM → audio_queue
- `moshi_loop`     — Moshi InferenceState → PCM send (priority) + tokens
- `bridge_loop`    — tokens → bridge (KV-cache) → audio_emb chunks
- `af_loop`        — audio_emb → AvatarForcing self-forcing → 4 frames/block
- `frame_encode`   — RGB → JPEG (TurboJPEG or cv2) → send_queue
- `keepalive`      — send_queue drain @ 25fps + 0xFE pings

**Port:** 7865 (RunPod proxied URL)


---
## 📦 Cell 1 — System Dependencies

In [ ]:
%%bash
apt-get update -qq && apt-get install -y -qq \
    ffmpeg libsndfile1 sox git wget curl 2>/dev/null
echo '✅ System deps installed'

## 📦 Cell 2 — Python: PyTorch

In [ ]:
!pip uninstall -y torch torchvision torchaudio flash-attn
!pip install torch==2.6.0 torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
print('✅ PyTorch 2.6 + CUDA 12.4 installed')

## 📦 Cell 3 — Python: Pipeline Dependencies

In [ ]:
%%bash
pip install -q \
    "moshi==0.2.4" \
    "sphn>=0.1.4" \
    "sentencepiece>=0.1.99" \
    "transformers==4.41.2" \
    "accelerate==0.31.0" \
    "diffusers==0.30.0" \
    "peft==0.13.0" \
    "ftfy" \
    "easydict" \
    "safetensors>=0.4.0" \
    "huggingface_hub>=0.23.0" \
    "omegaconf>=2.3.0" \
    "einops>=0.7.0" \
    "soundfile>=0.12.1" \
    "pyyaml>=6.0" \
    "pillow>=10.0" \
    "decord>=0.6.0" \
    "lmdb>=1.4.0" \
    "pandas>=2.0" \
    "opencv-python-headless>=4.8" \
    "pyworld>=0.3.4" \
    "tqdm>=4.66" \
    "ipywidgets>=8.0" \
    "numpy" \
    "scipy" \
    "imageio" \
    "imageio-ffmpeg" \
    "av" \
    "rotary-embedding-torch" \
    "pytorch-lightning" \
    "torchmetrics"
echo '✅ Pipeline deps installed'

## 📦 Cell 4 — Python: Streaming Server Dependencies

**Note:** `PyTurboJPEG<2.0` is pinned because v2.0 requires libjpeg-turbo ≥3.0  
which is not yet in most RunPod base images. v1.x works with libjpeg-turbo 2.x.  
`nest_asyncio` is required to run uvicorn inside a Jupyter kernel.

In [ ]:
%%bash
pip install -q \
    "fastapi>=0.110.0" \
    "uvicorn[standard]>=0.29.0" \
    "websockets>=12.0" \
    "python-multipart>=0.0.9" \
    "nest_asyncio>=1.6.0" \
    "PyTurboJPEG<2.0"
echo '✅ Streaming server deps installed'

## 📦 Cell 5 — Flash Attention

In [ ]:
!pip install https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.4.post1/flash_attn-2.7.4.post1+cu12torch2.6cxx11abiFALSE-cp311-cp311-linux_x86_64.whl --no-build-isolation

In [ ]:
import flash_attn; print(f'✅ flash_attn {flash_attn.__version__}')

---
## ⚙️ Cell 6 — Set Workspace Path

**Change `WORKSPACE` below to match your actual RunPod directory.**

In [ ]:
!git clone https://github.com/MoshiHead/Moshi-AvatarForcing-Streaming-v2.git

In [ ]:
import os, sys

# ── CHANGE THIS to your actual workspace path ─────────────────────────────────
WORKSPACE = '/workspace/Moshi-AvatarForcing-Streaming-v2'
# ─────────────────────────────────────────────────────────────────────────────

os.chdir(WORKSPACE)

MOSHI_ROOT  = f'{WORKSPACE}/moshi-inference'
BRIDGE_ROOT = f'{WORKSPACE}/moshi-wav2vec-bridge'
AF_ROOT     = f'{WORKSPACE}/AvatarForcing-inference'

for p in [WORKSPACE, MOSHI_ROOT, BRIDGE_ROOT, AF_ROOT]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f'✅ WORKSPACE = {WORKSPACE}')
print(f'   cwd = {os.getcwd()}')
print(f'   streaming_server exists = {os.path.isdir(WORKSPACE + "/streaming_server")}')

---
## 📥 Cell 7 — Download Model Weights

In [ ]:
import os
from huggingface_hub import snapshot_download, hf_hub_download

os.makedirs(f'{WORKSPACE}/checkpoints', exist_ok=True)

# ── Wan2.1-T2V-1.3B backbone ──────────────────────────────────────────────────
print('Downloading Wan2.1-T2V-1.3B …')
snapshot_download(
    repo_id   = 'Wan-AI/Wan2.1-T2V-1.3B',
    local_dir = f'{WORKSPACE}/AvatarForcing-inference/wan_models/Wan2.1-T2V-1.3B',
    ignore_patterns = ['*.md', '*.txt'],
)
print('✅ Wan2.1-T2V-1.3B ready')

In [ ]:
# ── AvatarForcing checkpoint ──────────────────────────────────────────────────
from huggingface_hub import snapshot_download
print('Downloading AvatarForcing checkpoints …')
snapshot_download(
    repo_id=  'lycui/AvatarForcing',
    local_dir=f'{WORKSPACE}/checkpoints',
    local_dir_use_symlinks=False,
)
print('✅ AvatarForcing checkpoint ready')

In [ ]:
# ── Bridge checkpoint ─────────────────────────────────────────────────────────
from huggingface_hub import hf_hub_download
hf_hub_download(
    repo_id   = 'Darknsu/new-bridge-model-12-layer-v1',
    filename  = 'bridge_best.pt',
    repo_type = 'dataset',
    local_dir = f'{WORKSPACE}/checkpoints',
)
BRIDGE_CKPT = f'{WORKSPACE}/checkpoints/bridge_best.pt'
AF_CKPT     = f'{WORKSPACE}/checkpoints/model.pt'
print(f'✅ Bridge: {BRIDGE_CKPT}  exists={os.path.exists(BRIDGE_CKPT)}')
print(f'✅ AF:     {AF_CKPT}      exists={os.path.exists(AF_CKPT)}')

---
## ⚙️ Cell 8 — Configure Environment

In [ ]:
import os

os.environ['BRIDGE_CKPT']   = f'{WORKSPACE}/checkpoints/bridge_best.pt'
os.environ['BRIDGE_CONFIG'] = f'{WORKSPACE}/moshi-wav2vec-bridge/config.yaml'
os.environ['AF_CKPT']       = f'{WORKSPACE}/checkpoints/model.pt'
os.environ['AF_CONFIG']     = f'{WORKSPACE}/AvatarForcing-inference/configs/avatarforcing.yaml'
os.environ['MOSHI_HF_REPO'] = 'kyutai/moshiko-pytorch-q8'
os.environ['DEVICE']        = 'cuda'
os.environ['PORT']          = '7865'
os.environ['HOST']          = '0.0.0.0'
os.environ['TEACHER_LEN']   = '80'
os.environ['USE_EMA']       = 'false'
os.environ['AF_PROMPT']     = (
    'A person talking naturally, realistic facial expressions, '
    'high quality video, detailed face.'
)

print('✅ Environment configured:')
for k in ['BRIDGE_CKPT','AF_CKPT','AF_CONFIG','DEVICE','PORT']:
    exists = ''
    if 'CKPT' in k or 'CONFIG' in k:
        exists = f'  [{"✅" if os.path.exists(os.environ[k]) else "❌ NOT FOUND"}]'
    print(f'  {k} = {os.environ[k]}{exists}')

---
## 🔍 Cell 9 — Verify Imports

In [ ]:
import torch
print(f'✅ torch {torch.__version__}, CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'   GPU: {torch.cuda.get_device_name(0)}')
    print(f'   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

from fastapi import FastAPI;    print('✅ fastapi')
import uvicorn;                 print('✅ uvicorn')
import nest_asyncio;            print('✅ nest_asyncio')
from omegaconf import OmegaConf; print('✅ omegaconf')
from einops import rearrange;   print('✅ einops')

try:
    from turbojpeg import TurboJPEG
    tj = TurboJPEG()
    print('✅ TurboJPEG (libjpeg-turbo)')
except Exception as e:
    print(f'⚠️  TurboJPEG: {e}')
    print('   → Will use cv2 fallback (still works, ~3× slower)')

print('\n✅ All imports verified — ready to launch.')

---
## 🌐 Cell 10 — Get RunPod Public URL
Run this before launching the server.

In [ ]:
import os
pod_id = os.environ.get('RUNPOD_POD_ID', 'YOUR_POD_ID')
print(f'🌐 Avatar UI:     https://{pod_id}-7865.proxy.runpod.net')
print(f'❤️  Health check: https://{pod_id}-7865.proxy.runpod.net/health')
print()
print('Steps after server starts:')
print('  1. Open the Avatar UI URL in your browser')
print('  2. Upload a frontal face image')
print('  3. Click "Start Session"')
print('  4. Click "Start Talking" and speak into your microphone')

---
## 🚀 Cell 11 — Launch Streaming Server (Port 7865)

> ⚠️ **This cell blocks while the server is running.**  
> Models load first (~1–3 min), then the server accepts connections.  
> Use `Kernel → Interrupt` to stop the server.

**Fix applied:** `nest_asyncio.apply()` patches the Jupyter event loop so  
uvicorn can run inside the notebook without the 'event loop already running' error.

In [ ]:
import nest_asyncio
nest_asyncio.apply()   # ← patches asyncio to allow nested event loops in Jupyter

import os, sys, uvicorn

# Make sure workspace is in sys.path
for p in [
    WORKSPACE,
    f'{WORKSPACE}/moshi-inference',
    f'{WORKSPACE}/moshi-wav2vec-bridge',
    f'{WORKSPACE}/AvatarForcing-inference',
]:
    if p not in sys.path: sys.path.insert(0, p)

os.chdir(WORKSPACE)

print('=' * 60)
print('  Real-Time AI Avatar Streaming Server')
print(f'  Port: 7865')
print('  Loading Moshi + Bridge + AvatarForcing …')
print('  (this takes 1–3 minutes on first run)')
print('=' * 60)

uvicorn.run(
    'streaming_server.server:app',
    host               = '0.0.0.0',
    port               = 7865,
    log_level          = 'info',
    ws_ping_interval   = 20,
    ws_ping_timeout    = 60,
    timeout_keep_alive = 30,
)

---
## 🔁 Alternative: Background Launch (non-blocking)

Run the server in a background thread so you can keep using the notebook.  
Use this if you want to run other cells while the server is active.

In [ ]:
# ── ALTERNATIVE: run server as background thread ─────────────────────────────
# Only use ONE of Cell 11 or this cell — not both.

import threading, nest_asyncio, uvicorn, os, sys
nest_asyncio.apply()

for p in [WORKSPACE, f'{WORKSPACE}/moshi-inference',
          f'{WORKSPACE}/moshi-wav2vec-bridge', f'{WORKSPACE}/AvatarForcing-inference']:
    if p not in sys.path: sys.path.insert(0, p)
os.chdir(WORKSPACE)

config = uvicorn.Config(
    'streaming_server.server:app',
    host='0.0.0.0', port=7865, log_level='info',
    ws_ping_interval=20, ws_ping_timeout=60, timeout_keep_alive=30,
)
server = uvicorn.Server(config)

thread = threading.Thread(target=server.run, daemon=True)
thread.start()

print('✅ Server starting in background thread.')
print('   Monitor logs below. Server will be ready once models finish loading.')
print(f'   URL: https://{os.environ.get("RUNPOD_POD_ID","YOUR_POD_ID")}-7865.proxy.runpod.net')

---
## ❤️ Cell 12 — Health Check

In [ ]:
import requests, json, time

print('Checking server health …')
for attempt in range(10):
    try:
        r = requests.get('http://localhost:7865/health', timeout=5)
        data = r.json()
        print(json.dumps(data, indent=2))
        break
    except Exception as e:
        print(f'  Attempt {attempt+1}/10: {e} — retrying in 10s …')
        time.sleep(10)
else:
    print('❌ Server not reachable. Make sure Cell 11 (or background launch) is running.')

---
## 🔧 Cell 13 — Bridge Shape Diagnostic

In [ ]:
# Verify bridge streaming output shape
import torch, yaml

sys.path.insert(0, f'{WORKSPACE}/moshi-wav2vec-bridge')
from model import MimiWav2Vec2Bridge
from streaming_server.pipeline.streaming_bridge import StreamingBridge

with open(f'{WORKSPACE}/moshi-wav2vec-bridge/config.yaml') as f:
    cfg = yaml.safe_load(f)

model = MimiWav2Vec2Bridge(cfg).to('cuda')
try:
    ckpt = torch.load(os.environ['BRIDGE_CKPT'], map_location='cuda', weights_only=True)
except TypeError:
    ckpt = torch.load(os.environ['BRIDGE_CKPT'], map_location='cuda')
sd = ckpt.get('bridge', ckpt)
model.load_state_dict(sd, strict=False)
model.eval()

bridge = StreamingBridge(model, device=torch.device('cuda'))
bridge.reset_session()

print('Simulating 4 streaming token pushes (2 flushes of BRIDGE_CHUNK=2):')
for i in range(4):
    r = bridge.push_token(torch.randint(0, 2048, (1, 8)), seq=i)
    if r is not None:
        seq_s, seq_e, emb = r
        print(f'  Flush [{seq_s},{seq_e}):  emb.shape={emb.shape}  expected=(4, 10752)')
        assert emb.shape == (4, 10752), f'Shape mismatch: {emb.shape}'

print('\n✅ Bridge streaming shape verified.')

---
## 📐 Architecture Reference

| Stage | Input | Output | Rate |
|---|---|---|---|
| Browser AudioWorklet | Microphone | int16 PCM (1920 samples) | 24 kHz |
| Moshi Mimi Encode | 1920-sample PCM | 8 codebook tokens | 12.5 Hz |
| Moshi Helium LM | 8 user tokens | 8 response tokens + 1 text | 12.5 Hz |
| Moshi Mimi Decode | 8 response tokens | 1920-sample PCM | 24 kHz |
| **Bridge (KV-cache, 2 tokens/flush)** | 2 Moshi tokens | (4, 10752) audio_emb | 25 Hz |
| **AvatarForcing self-forcing** | audio_emb + image KV-cache | 4 latent frames | 25 FPS |
| VAE Decode | 4 latents | 4 × (480, 832, 3) RGB | 25 FPS |
| TurboJPEG / cv2 | RGB frame | JPEG bytes | 25 FPS |
| WebSocket → Browser | JPEG + PCM | Lip-synced A+V | real-time |

**Key design choices:**
- `inference_self_forcing` — 1 block = 4 frames, KV-cache persists ALL blocks in a session
- Bridge KV-cache: 2 tokens/flush vs 40 tokens (batch) → **no 3.2s startup delay**
- Audio sent via priority `ws_write_lock` bypass — video never blocks audio
- Frame queue bounded at 30 — oldest frames dropped under GPU overload
- Audio_emb zero-padded when buffer is short — AF generates immediately
